# DLThon2 · 역할 A (문서 · 검색) — 코랩 실행

규약집 PDF → 쪽 단위 조각 · 표 행 문장 → BM25 + bge-m3 하이브리드 검색 → 판본 규칙 → 검색 품질 측정.
V3가 나오면 **맨 아래 'A-7' 절만** 다시 돌리면 됩니다.

| 절 | 업무 | 런타임 |
|---|---|---|
| 0 | 준비 (코드 · PDF · 설치) | CPU도 됨 |
| 1 | A-1~3 추출 · 표 행 · 청킹 + 추출 시험 | CPU |
| 2 | A-4 색인 (벡터는 GPU 권장, T4면 충분) | **GPU** |
| 3 | A-4 · A-6 검색과 판본 규칙 써 보기 | |
| 4 | A-5 검색 품질 | |
| 5 | A-7 V3 반영 · V2↔V3 차이 | |
| 6 | 결과를 드라이브에 저장 | |

**미리 할 것** : 구글 드라이브에 `MyDrive/dlthon2/pdf/` 폴더를 만들고 `rulebook_V2.pdf`를 넣는다. V3가 나오면 같은 폴더에 `rulebook_V3.pdf`.
PDF는 깃에 올리지 않는다(`.gitignore`).

## 0. 준비

In [2]:
# ── 여기만 고치세요 ─────────────────────────────────────────────
REPO_URL  = ""                                # 예: "https://github.com/<아이디>/dlthon2-rag.git"  (비우면 드라이브의 zip 사용)
DRIVE_DIR = "/content/drive/MyDrive/dlthon2"  # pdf/ 폴더가 있는 곳. 결과도 여기 out/ 에 저장
EMBED     = "BAAI/bge-m3"                     # 의미 검색 모델. "" 이면 BM25만
# ────────────────────────────────────────────────────────────────
import os, subprocess, glob, shutil
from google.colab import drive
drive.mount("/content/drive")

WORK = "/content/dlthon2-rag"
if not os.path.exists(WORK):
    if REPO_URL:
        url = REPO_URL
        try:   # 비공개 저장소면 코랩 왼쪽 '열쇠(보안 비밀)'에 GH_TOKEN 을 넣어 둔다
            from google.colab import userdata
            tok = userdata.get("GH_TOKEN")
            url = REPO_URL.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, WORK], check=True)
    else:
        zips = glob.glob(f"{DRIVE_DIR}/dlthon2-rag*.zip")
        assert zips, f"REPO_URL 이 비었고 {DRIVE_DIR} 에 dlthon2-rag*.zip 도 없습니다"
        subprocess.run(["unzip", "-q", "-o", zips[0], "-d", "/content"], check=True)
        if not os.path.exists(WORK):            # zip 안 폴더 이름이 다를 때
            top = [d for d in glob.glob("/content/*") if os.path.exists(f"{d}/rag/extract.py")]
            shutil.move(top[0], WORK)

os.makedirs(f"{WORK}/pdf", exist_ok=True)
for p in glob.glob(f"{DRIVE_DIR}/pdf/*.pdf"):
    shutil.copy(p, f"{WORK}/pdf/")
%cd {WORK}
print(sorted(os.listdir("pdf")))

Mounted at /content/drive
/content/dlthon2-rag
['rulebook_V2.pdf']


In [3]:
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 (벡터 만들기는 느림)"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Tesla T4, 15360 MiB


## 1. A-1 · A-2 · A-3 : PDF 추출 · 표 행 문장 · 쪽 단위 조각

- 본문과 표를 따로 뽑는다(표 영역은 본문에서 빼고 `pdfplumber` 표 인식). 머리말 · 꼬리말 제거
- 표의 각 행 → `표 41 가족행사별 공동비 상한 V2, 행사: …, 행사당 상한: 22만 원` (`rows.jsonl`)
- 쪽 하나 = 조각 하나. 판본 · 쪽 · 장 · 조항 · 표 · 참조 · 개정 여부 (`pages.jsonl`)
- 기대값(V2) : 쪽 60 · 표 행 342 · 경고 0건

In [4]:
!python -m rag.extract --doc V2
!head -12 data/V2/report.txt

[V2] 쪽 60 · 표 행 342 · 조항 111 · 표 60 -> /content/dlthon2-rag/data/V2
경고 0건
p01  조항 0  표 1  행 5  개정 -  신혼생활 운영위원회
p02  조항 0  표 1  행 12  개정 O  차례와 기준 찾기
p03  조항 2  표 1  행 5  개정 -  운영 범위와 역할
p04  조항 2  표 1  행 6  개정 -  금액과 시간
p05  조항 2  표 1  행 5  개정 -  기준 적용 순서
p06  조항 2  표 1  행 5  개정 -  고지 합의와 답변 상태
p07  조항 2  표 1  행 8  개정 -  사건 기준일과 판본
p08  조항 2  표 1  행 5  개정 -  가구 프로필과 월수입
p09  조항 2  표 1  행 9  개정 -  주택과 자산 부채
p10  조항 2  표 1  행 7  개정 -  통장과 연금 기록
p11  조항 2  표 1  행 10  개정 -  공동계좌 월 예산
p12  조항 2  표 1  행 6  개정 -  공동비와 개인비


**추출 시험** : 쪽 번호(꼬리말 대조) · 조항 · 표 번호 = 쪽 번호 · 칸 안 줄바꿈 복원(표 17 '같은 누수의 출동과 후속부품') · 프롬프트에 들어갈 본문

In [5]:
!python -m scripts.check_extract --doc V2 --pages 17 41 54 | head -80


V2-p17 · 제3장 공동생활비와 자산 관리 · 일반 수리 교체와 임시 조치
  [통과] 쪽 번호(꼬리말 대조)   꼬리말 17 / 조각 17
  [통과] 본문 조항          제29조, 제30조
  [통과] 본문 길이          339자
  [통과] 표 번호 = 쪽 번호    표 17
  [통과] 표 행 추출         5행
  [통과] 칸 안 줄바꿈 복원     모두 한 줄
  [통과] 알려진 행          같은 누수의 출동과 후속부품

--- 조각 본문 (프롬프트에 들어가는 그대로) ---
# 제3장 공동생활비와 자산 관리 · 일반 수리 교체와 임시 조치
안전에 문제가 없는 생활 수리에는 수리 한도와 일반 합의 기준을 적용한다.
## 제29조 일반 수리 상한
같은 원인의 출동·부품·시공 비용을 한 수리로 합산한다. 공동비 상한은 수리당 45만 원이며 표 11의 수리 가용액 또는 합의된 전용액을 사용한다. 20만 원 이상이면 제21조를 적용한다.
## 제30조 임시 조치와 본수리
정상적인 사용을 복구할 수 있을 때까지 같은 원인의 후속 작업을 한 수리로 관리한다. 긴급 최소조치에 쓴 비용도 수리 총액에는 포함한다. 비긴급 본수리의 추가 계약은 제22조에따라 다시 확인한다.
## 표 17 수리 범위
[표 17 수리 범위] 현상 | 처리 | 관련 기준
- 문손잡이 헐거움 | 일반 수리 | 제29조
- 센서 간헐 고장 | 일반 수리 | 제29조
- 같은 누수의 출동과 후속부품 | 같은 수리 건으로 합산 | 제23조·제30조
- 누수 봉쇄와 미관 복원 | 봉쇄만 긴급 검토 | 제31조
- 공동 현관의 비용 | 관리 주체 확인 후 가구 부담 판단 | 제41조
45만 원 이하라도 수리 예산이 모자라면 전용 합의가 필요하다.

--- 표 행 문장 (검색용 보조 조각) ---
  표 17 수리 범위, 현상: 문손잡이 헐거움, 처리: 일반 수리, 관련 기준: 제29조
  표 17 수리 범위, 현상: 센서 간헐 고장, 처리: 일반 수리, 관련 기준: 

## 2. A-4 : 색인

조각 402개(쪽 60 + 표 행 342). BM25는 검색할 때 만들고, 여기서는 벡터(`emb.npy`)만 만든다. T4 기준 bge-m3 첫 다운로드 포함 몇 분.

In [6]:
if EMBED:
    !python -m rag.index --doc V2 --embed {EMBED}
else:
    !python -m rag.index --doc V2

조각 402개 (쪽 60 + 표 행 342)
modules.json: 100% 349/349 [00:00<00:00, 1.79MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 729kB/s]
README.md: 100% 15.8k/15.8k [00:00<00:00, 45.7MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00<00:00, 255kB/s]
config.json: 100% 687/687 [00:00<00:00, 4.03MB/s]

pytorch_model.bin: downloading bytes:  10% 226M/2.27G [00:01<00:09, 219MB/s, 16.6MB/s  ]  
pytorch_model.bin: reconstructing file:   3% 67.1M/2.27G [00:02<01:05, 33.5MB/s]
pytorch_model.bin: downloading bytes:  31% 701M/2.27G [00:02<00:03, 522MB/s, 57.0MB/s  ] ]
pytorch_model.bin: downloading bytes:  40% 911M/2.27G [00:03<00:02, 627MB/s, 73.4MB/s  ] ]
pytorch_model.bin: downloading bytes:  55% 1.24G/2.27G [00:03<00:01, 773MB/s,  100MB/s  ]]
pytorch_model.bin: downloading bytes:  65% 1.48G/2.27G [00:03<00:01, 660MB/s,  122MB/s  ] ]
pytorch_model.bin: reconstructing file:  71% 1.61G/2.27G [00:04<00:01, 529MB/s, 93.0MB/s  ]
pytorch_model.bin: reconstructing file:  92% 2.08G/2.27G

## 3. 검색 · 판본 규칙 써 보기 (A-4 · A-6)

`[판본]` 줄 : 표 07 항목 → 기준일 종류 → 질문 속 날짜 → 부칙 제1조 시행 구간. 날짜가 없으면 `추가 확인`(되묻기).
쪽 줄의 `[ ]` 안 : 검색 순위 · 표 60 색인어 · 참조 따라가기 · 판본 규칙으로 붙은 이유.

In [7]:
!python -m rag.search "10월 15일에 시어머니 생신 선물 15만 원과 식사 7만 원을 서로 다른 카드로 결제할 예정이야" --doc V2
print()
!python -m rag.search "어머님 생신 식사와 선물을 서로 다른 카드로 결제했어. 공동비로 처리할 수 있을까?" --doc V2

Loading weights: 100% 391/391 [00:00<00:00, 19233.38it/s]
[판본] 확정 · 항목 가족행사 공동비 상한 · 기준 행사 첫 확정 결제일 · 날짜 2026-10-15 → V2
       행사 첫 확정 결제일 2026-10-15 → V2 (시행 구간 V1 2026-06-01~, V2 2026-10-01~)
p41 생신 식사와 선물          제77조 · 제78조 · 표 41  [검색 1위; 색인어 '생신'(표 60)]  행: 표 41 · 시부모·장인·장모 등 부모님 생신, 표 41 · 명절 방문과 선물, 표 41 · 양가 일반 식사
p54 가족행사와 가전 개정        부칙 제3조 · 부칙 제4조 · 표 54  [검색 2위; 색인어 '생신'(표 60)]  행: 표 54 · 부모님 생신 행사당 상한, 표 54 · 가족행사 사전 합의, 표 54 · 가전 사전 합의
p42 행사 예산과 돌봄 교통비      제79조 · 제80조 · 표 42  [검색 3위; 색인어 '생신'(표 60)]  행: 표 42 · 생신 식사와 케이크, 표 42 · 월 가용액 부족, 표 42 · 별도 가족 생신 두 건
p59 여러 근거를 연결하는 공개 사례  표 59  [검색 4위]  행: 표 59 · V2, 생신 선물 15만 원과 식사 7만 원을 다른 카드로 결제 예정. 월 잔액 충분, 표 59 · 전세대출 정리 후 4.88%에 30년 26,400만 원을 빌리는시나리오
p58 개정 전과 후의 공개 사례     표 58  [검색 5위]  행: 표 58 · 생신 식사와 케이크 24만 원, 표 58 · 호텔 1박 14만원
p14 거래 합산과 할인          제23조 · 제24조 · 표 14  [참조 : p54의 제23조, p42의 제23조, p42의 제24조]
p50 회식 술자리와 귀가 약속      제95조 · 제96조 · 표 50  [참조 : p59의 제95조, p59의 제96조, p59의 표 50]
p31 추가매수 단계           

In [8]:
# 경과조치 : 환불은 원거래 판본, 전량 취소 후 새 주문은 새 날짜
!python -m rag.version "공용 밥솥을 35만 원에 9월 30일 주문 확정했고 10월 3일에 환불받았어" --doc V2 | grep -E "applied|reason"
!python -m rag.version "9월 28일 주문한 로봇청소기를 전량 취소하고 10월 2일 새 주문으로 확정했어" --doc V2 | grep -E "applied|reason"

 "applied_version": "V1",
 "reason": "최초 확정 주문일 2026-09-30 → V1 (시행 구간 V1 2026-06-01~, V2 2026-10-01~). 환불은 원거래 판본 유지(제10조 · 제26조)",
 "applied_version": "V2",
 "reason": "최초 확정 주문일 2026-10-02 → V2 (시행 구간 V1 2026-06-01~, V2 2026-10-01~). 전량 취소 후 새 주문 · 예약은 새 날짜(부칙 제4조 · 제7조 · 제9조)",


## 4. A-5 : 검색 품질

정답 조항 · 표가 있는 쪽이 결과에 들어왔는지. 기대값(V2, BM25만) : 검색만 0.902 · 검색+참조+판본 0.985 · 판본 16/16.
벡터를 더하면 검색만 값이 달라진다(RunPod bge-m3 실측 0.931).
팀 질문 세트가 생기면 `--questions eval/questions_team.jsonl`.

In [9]:
!python -m eval.eval_retrieval --doc V2

Loading weights: 100% 391/391 [00:00<00:00, 13294.53it/s]
[검색만         ] k=5  page_recall 0.931  all_hit 14/17  mrr 0.824
[검색+참조+판본    ] k=5  page_recall 0.985  all_hit 16/17  mrr 0.824  version_acc 16/16  (평균 쪽 수 10.4)

놓친 쪽 (검색+참조+판본)
  T59-2   놓친 쪽 [4]

-> /content/dlthon2-rag/results/retrieval_V2_20261117-1422.jsonl


**판본 규칙만 채점 (v2)** : 판본을 헷갈리게 만든 30문항 + 처음 보는 15문항. 기대값 30/30 · 15/15

In [ ]:
!python -m eval.eval_version --doc V2
!python -m eval.eval_version --doc V2 --questions eval/questions_version_holdout.jsonl
!python -m eval.eval_version --doc V2 --questions eval/questions_version_team.jsonl
!python -m eval.eval_version --doc V2 --questions eval/questions_version_dev50.jsonl

## 5. A-7 : V3 반영

V3 PDF를 `DRIVE_DIR/pdf/rulebook_V3.pdf`로 넣고 **0절 두 번째 셀(PDF 복사)부터** 다시 실행한 뒤 아래를 순서대로.

1. 추출 → `report.txt` 경고 확인
2. `add_version` (확인만) : 시행일 · 표 07 항목 · 대조표 쪽 · 구조(표 번호 = 쪽 번호, 머리말 · 꼬리말)를 **자동으로 찾아 보여 줌**
3. 내용이 맞으면 `--write`로 `config/versions.json`에 반영. **새 항목**이 나오면 `items`의 `any`에 키워드를 채움
4. 색인 → 품질 → V2↔V3 차이(`results/diff_V2_V3.md`, C에게 '개정 대응 질문' 재료로)

문서마다 '아는 판본'이 다르다 : 같은 12월 날짜 질문도 `--doc V2`면 V2, `--doc V3`면 V3로 판정한다. 이것이 V2/V3 비교 실험의 축.

In [10]:
V3_PDF = "pdf/rulebook_V3.pdf"
assert os.path.exists(V3_PDF), "V3 PDF가 아직 없습니다 (드라이브 pdf/ 에 넣고 0절 PDF 복사 셀을 다시 실행)"
!python -m rag.extract --doc V3 --pdf {V3_PDF}
!python -m scripts.add_version --doc V3 --pdf {V3_PDF}

AssertionError: V3 PDF가 아직 없습니다 (드라이브 pdf/ 에 넣고 0절 PDF 복사 셀을 다시 실행)

위 `[확인할 것]`이 비었고 시행일 · 항목이 PDF와 맞으면 반영합니다. 시행일을 못 찾았다면 `--start 2026-MM-DD`를 붙입니다.

In [ ]:
!python -m scripts.add_version --doc V3 --pdf {V3_PDF} --write
if EMBED:
    !python -m rag.index --doc V3 --embed {EMBED}
else:
    !python -m rag.index --doc V3
!python -m eval.eval_retrieval --doc V3
!python -m eval.eval_version --doc V3
!python -m scripts.diff_versions --old V2 --new V3 --questions eval/questions_seed.jsonl

## 6. 결과를 드라이브에 저장
`config/versions.json`이 바뀌었으면(V3 반영) 로컬 저장소에 복사해 커밋합니다.

In [11]:
import time
dst = f"{DRIVE_DIR}/out/{time.strftime('%m%d-%H%M')}"
os.makedirs(dst, exist_ok=True)
for d in ("results", "data", "config"):
    if os.path.exists(d):
        shutil.copytree(d, f"{dst}/{d}", dirs_exist_ok=True, ignore=shutil.ignore_patterns("emb.npy"))
print("->", dst)

-> /content/drive/MyDrive/dlthon2/out/1117-1425
